<a href="https://colab.research.google.com/github/lathikasuriyamoorthy-29/trendpulse-lathikasuriyamoorthy/blob/main/TrendPulse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import time
import json
import os
from datetime import datetime


# --------------------------------------------------
# 1. API URLs
# --------------------------------------------------

TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEMS_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"


# --------------------------------------------------
# 2. Categories and keywords
# --------------------------------------------------

categories = {
    "technology": [
        "AI", "software", "tech", "code",
        "computer", "data", "cloud", "API", "GPU",
        "LLM"
    ],

    "worldnews": [
        "war", "government", "country", "president",
        "election", "climate", "attack", "global"
    ],

    "sports": [
        "NFL", "NBA", "FIFA", "sport", "sports",
        "team", "player", "league", "championship",

    ],

    "science": [
        "research", "study", "space", "physics",
        "biology", "discovery", "NASA", "genome",

    ],

    "entertainment": [
        "movie", "film", "music", "Netflix", "game","book",
        "show", "award", "streaming"
    ]
}


# --------------------------------------------------
# 3. Function to classify a story
# --------------------------------------------------

def get_category(title):

    title_lower = title.lower()

    for category, keywords in categories.items():

        for keyword in keywords:

            if keyword.lower() in title_lower:
                return category

    return None


# --------------------------------------------------
# 4. Get the first 500 top-story IDs
# --------------------------------------------------

response = requests.get(TOP_STORIES_URL)

story_ids = response.json()[:1000]

print("Story IDs fetched:", len(story_ids))


# --------------------------------------------------
# 5. Storage
# --------------------------------------------------

collected_stories = []

category_counts = {
    "technology": 0,
    "worldnews": 0,
    "sports": 0,
    "science": 0,
    "entertainment": 0
}


# --------------------------------------------------
# 6. Collect stories category by category
# --------------------------------------------------

for category in categories:

    print("\nCollecting:", category)

    for story_id in story_ids:

        # Stop when this category reaches 25
        if category_counts[category] >= 25:
            break

        # Create URL for this particular story
        story_url = ITEMS_URL.format(story_id)

        try:
            story_response = requests.get(
                story_url,
                timeout=10
            )

            story = story_response.json()

        except requests.RequestException:
            continue

        # to make sure we received a story
        if not story or story.get("type") != "story":
            continue

        # Get title
        title = story.get("title", "")

        # Determine category
        detected_category = get_category(title)

        # Skip if it doesn't belong to the category
        # currently being collected
        if detected_category != category:
            continue

        # to create our own structured dictionary for the story informations
        story_data = {
            "post_id": story.get("id"),
            "title": title,
            "category": category,
            "score": story.get("score", 0),
            "num_comments": story.get("descendants", 0),
            "author": story.get("by"),
            "collected_at": datetime.now().isoformat()
        }

        # Store the story
        collected_stories.append(story_data)

        # Increase category count
        category_counts[category] += 1

    # Wait 2 seconds before starting next category
    time.sleep(2)


# --------------------------------------------------
# 7. Display results
# --------------------------------------------------

print("\n-----------------------------")
print("COLLECTION COMPLETE")
print("-----------------------------")

print("Total stories:", len(collected_stories))

print("\nCategory counts:")

for category, count in category_counts.items():
    print(category, ":", count)


# --------------------------------------------------
# 8. Save as JSON
# --------------------------------------------------

os.makedirs("data", exist_ok=True)

filename = (
    "data/trends_"
    + datetime.now().strftime("%Y%m%d")
    + ".json"
)

with open(filename, "w", encoding="utf-8") as file:

    json.dump(
        collected_stories,
        file,
        indent=4,
        ensure_ascii=False
    )

print("\nJSON file saved as:")
print(filename)

Story IDs fetched: 500

Collecting: technology

Collecting: worldnews

Collecting: sports

Collecting: science

Collecting: entertainment

-----------------------------
COLLECTION COMPLETE
-----------------------------
Total stories: 74

Category counts:
technology : 25
worldnews : 12
sports : 2
science : 10
entertainment : 25

JSON file saved as:
data/trends_20260928.json
